# Phase 3 — Scalable Ingestion

**Weeks 7–9 · Goal:** Ingest hundreds of PDFs efficiently with caching, deduplication, and observability.

## What you will learn

- **Embedding cache** — Redis stores vectors keyed by content fingerprint (skip re-embed on unchanged chunks)
- **Document fingerprint** — skip entire PDFs when file hash unchanged
- **MinHash dedup** — near-duplicate chunk detection before upsert
- **Celery workers** — async ingest queue for bulk jobs
- **Prometheus metrics** — throughput, latency, cache hit rate on `:9100/metrics`

> **Code:** `phases/phase_03_scalable_ingestion/`


In [1]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


Project root: C:\Users\dyh\Desktop\RAG


## Ingest modes

`IngestMode` controls parser/embedding cost:

| Mode | Parsers | Use case |
|------|---------|----------|
| `TEXT_ONLY` | text + tables | Fast bulk indexing |
| `MULTIMODAL` | + figures + pages | Full multimodal |
| `COLPALI` | + ColPali page embeds | GPU page-level retrieval |


In [2]:
from phases.phase_03_scalable_ingestion.pipeline.ingest_modes import IngestMode, build_ingestion_pipeline

for mode in IngestMode:
    pipeline = build_ingestion_pipeline(mode)
    parser_names = [type(p).__name__ for p in pipeline.parsers]
    print(f"{mode.value:12} parsers: {parser_names}")


full         parsers: ['PDFTextParser', 'TableParser', 'FigureParser', 'PageImageParser']
text_only    parsers: ['PDFTextParser', 'TableParser']


In [3]:
from phases.phase_03_scalable_ingestion.embedding.fingerprint import chunk_embed_key, doc_fingerprint
from shared.models import ChunkType, DocumentChunk, DocumentType

chunk = DocumentChunk(
    id="c1",
    doc_id="d1",
    source_path="x.pdf",
    doc_type=DocumentType.PDF,
    chunk_type=ChunkType.TEXT,
    content="Revenue grew 15% year-over-year.",
    page_number=1,
)
print("Chunk embed key:", chunk_embed_key(chunk))
print("Doc fingerprint:", doc_fingerprint(ROOT / "README.md"))


Chunk embed key: 5af07c28c46908111192c10d85fea9bcc4d30e227a3069f3e0221f953d77e7fe
Doc fingerprint: e1435fb450843c26


In [4]:
# Redis cache demo (requires Redis on localhost:6379)
try:
    from phases.phase_03_scalable_ingestion.embedding.redis_cache import EmbeddingCache

    cache = EmbeddingCache()
    content_hash = "demo123"
    cache.set_vector(content_hash, "bge-m3", [0.1, 0.2, 0.3])
    hit = cache.get_vector(content_hash, "bge-m3")
    print(f"Cache hit: {hit is not None}, dim={len(hit) if hit else 0}")
except Exception as exc:
    print(f"Redis unavailable ({exc}). Start: cd docker && docker compose up -d redis")


Redis unavailable (Install redis: pip install redis). Start: cd docker && docker compose up -d redis


## Bulk ingest CLI

```powershell
cd docker
docker compose --profile phase3 up -d qdrant redis ingest-worker
docker compose exec ingest-worker python phases/phase_03_scalable_ingestion/bulk_ingest.py \
  --glob "data/raw/bulk/*.pdf" --sync --text-only --hybrid --recursive-chunk
```

Generate 100 test PDFs: `python scripts/generate_bulk_pdfs.py --count 100`

**Next:** [04_taxonomy_generation.ipynb](04_taxonomy_generation.ipynb)
